In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/Wildlife_Export_2023_2025.csv', encoding='latin-1')

### Stripping the Column Headers

In [ ]:
df.columns = df.columns.str.strip()

### Getting More Information About the Data
* There are 65,774 records for the time period between 2023 & 2025
* This dataset has 102 columns, much of this data can probably be dropped.

In [ ]:
df.shape

### Checking the datatype for all columns
<u>Issues with the datatypes</u>
* `INCIDENT_DATE` is a string
* `TIME` is a string
* `NR_INJURIES` is a float
* `NR_FATALITIES` is a float

In [67]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 65774 entries, 0 to 65773
Data columns (total 103 columns):
 #    Column                 Non-Null Count  Dtype         
---   ------                 --------------  -----         
 0    INDX_NR                65774 non-null  int64         
 1    INCIDENT_DATE          65774 non-null  datetime64[us]
 2    INCIDENT_MONTH         65774 non-null  int64         
 3    INCIDENT_YEAR          65774 non-null  int64         
 4    TIME                   65774 non-null  str           
 5    TIME_OF_DAY            34364 non-null  str           
 6    AIRPORT_ID             65774 non-null  str           
 7    AIRPORT                65774 non-null  str           
 8    AIRPORT_LATITUDE       53792 non-null  object        
 9    AIRPORT_LONGITUDE      53789 non-null  object        
 10   RUNWAY                 65774 non-null  str           
 11   STATE                  53792 non-null  str           
 12   FAAREGION              53792 non-null  str           
 

### Checking how much of the data is null
* The following code creates the variable `null_pct` which will return the percentage of data that is missing from each column.
* The print statement will return all columns that have over **50%** of the data missing.
* I will consider this information in choosing which columns to keep.

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False) * 100
print(null_pct[null_pct > 50])

### Converting `INCIDENT_DATE` to `datetime`
* This code will convert `INCIDENT_DATE` to `datetime` which will provide for more robust data. 

In [ ]:
df['INCIDENT_DATE'] = pd.to_datetime(df['INCIDENT_DATE'])

### Comparing TIME and TIME_OF_DAY
* `TIME` has Zero nulls
* `TIME_OF_DAY` is missing 47% of the data

In [ ]:
print(df['TIME'].isna().sum(), df['TIME'].isna().mean())
print(df['TIME_OF_DAY'].isna().sum(), df['TIME_OF_DAY'].isna().mean())

### Considering and Adjusting `NR_INJURIES` and `NR_FATALITIES`
* `NR_INJURIES` and `NR_FATALITIES` are both floats. These should be converted to ints because a person is either injured or not.
* `NR_INJURIES` is 99.96% empty and `NR_FATALITIES` is 99.99% empty, but the data that does exist is still interesting enough to keep at this time.

In [ ]:
df['NR_INJURIES'] = df['NR_INJURIES'].fillna(0).astype(int)

In [ ]:
df['NR_FATALITIES'] = df['NR_FATALITIES'].fillna(0).astype(int)

### Checking for Duplicates
* There are no duplicate rows

In [ ]:
df.duplicated().sum()

### Initial Observations About the Data
* `INCIDENT_YEAR` The number of bird strikes is increasing year-over-year from 2023 to 2025.
* `INCIDENT_MONTH` Most bird strikes are recorded in September, August, October and July. There seems to be a significant difference between the warmer and colder months.
* `STATE` The top 5 states are TX, FL, CA, CO and TN. This seems to track with where the busy airports are (perhaps I can bring some data in regarding airport activity). Kentuky made the top 10.
* `FAAREGION` There seems to be some duplication in the FAAREGION data. I will need to strip this column to normalize it.
* `AIRPORT` UKNOWN is by far the largest airport, I am curious why that is the case. My guess is that it because many of the bird strikes are from observation of evidence of a bird strike but a lack of clarity where it happened. The other airports make sense because they are large market hubs.
* `OPERATOR` Again, UNKNOWN is by far the largest operator followed by the other popular airlines. I do find it interesting that business beats Delta and United, as those are very popular airlines.
* `PHASE_OF_FLIGHT` This is interesting because it describes at what point a plane is most likely to have a bird strike. I will need to supliment my understanding of these phases of flight because I do not know what makes them distinct.




In [ ]:
df['INCIDENT_YEAR'].value_counts()

In [ ]:
df['INCIDENT_MONTH'].value_counts()

In [ ]:
df['STATE'].value_counts().head(10)

In [ ]:
df['FAAREGION'].value_counts().head(10)

In [ ]:
df['AIRPORT'].value_counts().head(10)

In [ ]:
df['OPERATOR'].value_counts().head(10)

In [ ]:
df['NR_FATALITIES'].value_counts()

In [ ]:
df['PHASE_OF_FLIGHT'].value_counts().head(10)

In [ ]:
df['TIME_OF_DAY'].value_counts().head(10)

In [ ]:
df['TIME_OF_DAY'].isna().sum()

In [ ]:
df['TIME_OF_DAY'].value_counts(dropna=False)

In [ ]:
df['TIME'].head(10)

In [ ]:
df['INGESTED_OTHER'].value_counts()

In [ ]:
df['ENG_1_POS'].value_counts()

### Exploring the Danger of Bird Strikes
The original dataset has two fields to display number of injuries and deaths due to FAA bird strikes. I will create a new dataframe to just include columns that add context to the strikes that caused injury or death called `df_human_impact`.
* On January 1, 2024, there was an incident where 3 individuals died after striking a Cackling Goose.
* There are 22 injuries recorded between spanning between 2023-2025, one of them being a white-tailed deer.


In [ ]:
df_human_impact = df[['INCIDENT_DATE','AIRCRAFT','NR_INJURIES','NR_FATALITIES','SPECIES','ENROUTE_STATE']]

In [ ]:
df_human_impact[df_human_impact['NR_FATALITIES']>0]

In [ ]:
df_human_impact[df_human_impact['NR_INJURIES']>0].sort_values('NR_INJURIES', ascending=False)

### Exploration of Birds
* Unknown bird and unknown bird - small are the two most observed bird strikes, my assumption is that these birds are unknown because it happens in the air or the damage to the bird is so severe that there is nothing left to record. The other birds are common birds that are found throughout the country.
* `BIRD_BAND_NUMBER` is interesting but looking at more information online I can not really see a good resource to get any additional information on it. I will probably drop this column.
* `NUM_STRUCK` I am surprised that there are so many instances of 11-100 and more than 100 birds being struck. That defies my expecations.
* `SIZE` The majority of birds being struck are small, medium and then large. This meets my expecations.

In [ ]:
df['SPECIES'].value_counts().head(10)

In [ ]:
df['BIRD_BAND_NUMBER'].value_counts().head()

In [ ]:
df['NUM_STRUCK'].value_counts()

In [ ]:
df['SIZE'].value_counts()

### Exploring Species and Size of Bird
* The below code will create a variable of all `Largr`, `Medium`, and `Small` birds.

In [ ]:
df_LGBIRD = df[df['SIZE'] == 'Large']

In [ ]:
df_MDBIRD = df[df['SIZE'] == 'Medium']

In [ ]:
df_SMBIRD = df[df['SIZE'] == 'Small']

### Top 10 Large, Medium and Small Birds

In [ ]:
df_LGBIRD['SPECIES'].value_counts().head(10)

In [ ]:
df_MDBIRD['SPECIES'].value_counts().head(10)

In [ ]:
df_SMBIRD['SPECIES'].value_counts().head(10)

### Cleaning the Data
* After my initial observations of the data - I have decided to eliminate the following columns:
* <u>REDUNDANT DATE FIELDS</u>
    * `INCIDENT_MONTH`
    * `INCIDENT_YEAR`
* <u>ADMINISTRATIVE COLUMNS</u>
    * `INDX_NR` - Database index.
    * `LUPDATE` - last update timestamp
    * `TRANSFER` - Transfer tag
    * `IMAGE` - Boolean for whether a phot was taken, not useful.
    * `REPORTER_NAME` - Mostly redacted
    * `REPORTER_TITLE` - Mostly redacted
    * `PERSON` - Redacted
    * `FLT`
    * `REG`
* <u>MISSING DATA</u>
    * `BIRD_BAND_NUMBER`
    * `ENG_4_POS`
    * `COST_REPAIRS`
    * `EFFECT_OTHER`
    * `ENG_3_POS`
    * `COST_OTHER_INFL_ADJ`
    * `COST_OTHER`
    * `ENROUTE_STATE`
    * `AOS`
    * `OTHER_SPECIFY`
    * `LOCATION`
    * `INJESTED_OTHER`
* <u>AIRCRAFT MAKE/MODEL CODES MADE REDUNDANT BY `AIRCRAFT`</u>
    * `AMA`
    * `AMO`
    * `EMA`
    * `EMO`

In [70]:
df_clean = df.drop(['INCIDENT_MONTH','INCIDENT_YEAR','INDX_NR','LUPDATE','TRANSFER','IMAGE','REPORTER_NAME','REPORTER_TITLE','PERSON','FLT','REG','BIRD_BAND_NUMBER','ENG_4_POS','COST_REPAIRS','EFFECT_OTHER','ENG_3_POS','COST_OTHER_INFL_ADJ','COST_OTHER','ENROUTE_STATE','AOS','OTHER_SPECIFY','LOCATION','INGESTED_OTHER','AMA','AMO','EMA','EMO','NR_INJURIES', 'NR_FATALITIES'], axis=1)

In [71]:
df_clean.head(10)

,INCIDENT_DATE,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,AIRPORT_LATITUDE,AIRPORT_LONGITUDE,RUNWAY,STATE,FAAREGION,...,REMARKS,REMAINS_COLLECTED,REMAINS_SENT,WARNED,NUM_SEEN,NUM_STRUCK,SIZE,COMMENTS,SOURCE,TIME_MINUTES
0,2023-01-01,,Night,KJFK,JOHN F KENNEDY INTL,40.63975,-73.77893,22L,NY,AEA,...,RADOME DENT RH SIDE. Flight #: LAE2516 (Latam)...,False,False,Unknown,,1,Large,NaN,Multiple,None
1,2023-01-01,13:49,Day,KLGA,LA GUARDIA ARPT,40.77724,-73.87261,4,NY,AEA,...,1349L ATCT reported UAL795 struck a bird on de...,False,False,Unknown,,1,NaN,NaN,Multiple,None
2,2023-01-01,7:15,NaN,KMOB,MOBILE REGIONAL,30.69142,-88.24283,33,AL,ASO,...,"ENY3864, E170, DEPARTED RWY 33 AND REPORTED A ...",False,False,Unknown,,1,NaN,Revised POF,MOR,None
3,2023-01-01,14:10,NaN,KBOS,GENERAL EDWARD LAWRENCE LOGAN INTL ARPT,42.36435,-71.00518,33L,MA,ANE,...,Pilot departing 33L reported seeing a dead bir...,True,False,Unknown,,1,Medium,NaN,FAA Form 5200-7-E,None
4,2023-01-01,8:50,NaN,KBZN,GALLATIN FIELD ARPT,45.7769,-111.15301,12,MT,ANM,...,No aircraft/pilot reported damage or a strike....,True,False,Unknown,,1,Small,NaN,FAA Form 5200-7-E,None
5,2023-01-01,10:12,NaN,KATL,HARTSFIELD - JACKSON ATLANTA INTL ARPT,33.64044,-84.42694,Taxiway Bravo,GA,ASO,...,"During the daily airfield inspection, ATL Airp...",True,False,Unknown,,1,Small,NaN,FAA Form 5200-7-E,None
6,2023-01-01,10:30,NaN,KIAH,GEORGE BUSH INTERCONTINENTAL/ HOUSTON ARPT,29.98047,-95.33972,8/26,TX,ASW,...,Found during morning RWY inspection. day .DAY.,True,False,Unknown,,1,Small,NaN,FAA Form 5200-7-E,None
7,2023-01-01,8:50,NaN,KTPA,TAMPA INTL,27.97547,-82.53325,1L,FL,ASO,...,Remains located on center during SMGCS inspect...,True,True,Unknown,,1,Small,NaN,FAA Form 5200-7-E,None
8,2023-01-01,10:30,NaN,KAUS,AUSTIN-BERGSTROM INTL,30.19453,-97.66987,18L,TX,ASW,...,"During routine inspection, AUS Airside Operati...",True,False,Unknown,,1,Medium,NaN,FAA Form 5200-7-E,None
9,2023-01-01,20:30,NaN,KSEA,SEATTLE-TACOMA INTL,47.44898,-122.30931,16L,WA,ANM,...,found on runway check. NIGHT,True,False,Unknown,,1,Medium,corrected spelling in remarks,FAA Form 5200-7-E,None
